# FD004 IID rerun: standard (IID) evaluation, five seeds, converged training

Symmetry run for Paper 1. The DS-A rerun (`06_FD004_DS_A_Rerun_Converged_FiveSeed.ipynb`) trains under a pre-specified,
training-only convergence rule. This notebook applies **the same rule** to the IID protocol so that IID, DS-A and DS-B are
reported on one training protocol, five seeds each (42, 123, 456, 789, 1000), seed 42 as the single-run reference.

Unchanged from the frozen IID pipeline: data split, preprocessing, windows, architecture (20→18 projection, BiLSTM(64),
temporal attention, q10/q50/q90 pinball loss), Adam 1e-3, batch 256, and every calibration rule.

Changed (Part A, written before any training): 20% of training engines (split seed 42) become internal validation; up to
150 epochs; early stopping on internal-validation pinball loss, patience 15, not before epoch 30, best weights restored;
converged if best validation loss ≤ 0.60 × constant-quantile loss. Every seed is reported; none is replaced.

Part B trains and saves all models without touching evaluation windows; Part C evaluates afterwards.
Set `FD004_SMOKE=1` for a 1-seed, 3-epoch pipeline check.

In [1]:
import os, sys, json, time, random, hashlib, platform
from pathlib import Path

SMOKE_TEST = os.environ.get("FD004_SMOKE") == "1"

def _find_root():
    for c in [os.environ.get("FD004_ROOT", ""), "/content/drive/MyDrive/FD004_Research", str(Path.home() / "Downloads/FD004_Research")]:
        if c and Path(c, "CMAPSSData/train_FD004.txt").exists():
            return Path(c)
    raise FileNotFoundError("Set FD004_ROOT to the FD004_Research folder.")

if "google.colab" in sys.modules and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = _find_root()
SLUG = "IID"
CONDITION = "IID"

OUT = Path(os.environ.get("FD004_OUT") or ROOT / (f"FD004_{SLUG}_Rerun_Results" + ("_SMOKE" if SMOKE_TEST else "")))
(OUT / "models").mkdir(parents=True, exist_ok=True)
(OUT / "histories").mkdir(exist_ok=True)
print("Root:", ROOT, "| Output:", OUT)

Root: <FD004_ROOT> | Output: <FD004_ROOT>/FD004_IID_Rerun_Results


## Part A: protocol (written before any training or evaluation)

In [2]:
PROTOCOL = {
    "experiment": f"FD004 {CONDITION} converged five-seed rerun (symmetry with DS-A rerun)",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "data_seed": 42,
    "model_seeds": [42] if SMOKE_TEST else [42, 123, 456, 789, 1000],
    "internal_validation": {"fraction_of_train_engines": 0.20, "split_seed": 42},
    "model": {"projection": 18, "bilstm_units": 64, "quantiles": [0.10, 0.50, 0.90], "learning_rate": 1e-3, "batch_size": 256},
    "training_rule": {"max_epochs": 3 if SMOKE_TEST else 150, "monitor": "val_loss", "patience": 15,
                      "start_from_epoch": 0 if SMOKE_TEST else 30, "restore_best_weights": True},
    "convergence_check": {"ratio": 0.60, "reference": "constant training quantiles evaluated on internal validation",
                          "failed_seeds": "reported and flagged; never replaced or dropped"},
    "calibration": {"alpha": 0.20, "methods": ["Raw Quantiles", "Window CQR", "Asymmetric CQR",
                                               "Regime-conditional Asymmetric CQR", "Balanced Engine CQR"],
                    "window": "finite-sample higher quantile at ceil((n+1)(1-alpha))/n",
                    "asymmetric": "per tail at ceil((n+1)(1-alpha/2))/n, unclipped scores",
                    "regime_conditional": "asymmetric rule within each endpoint regime",
                    "balanced": "within-engine np.quantile(.,0.80), then ceil((M+1)(1-alpha))/M higher",
                    "interval_clip_at_zero": True},
    "life_stage": {"Early": "RUL > 80", "Mid": "40 < RUL <= 80", "Late": "RUL <= 40"},
    "bootstrap": {"replicates": 10000, "unit": "evaluation engine", "rng_seed": 42},
}
protocol_path = OUT / "protocol.json"
if protocol_path.exists() and not SMOKE_TEST:
    old = json.loads(protocol_path.read_text())
    changed = {k for k in PROTOCOL if k != "created_utc" and old.get(k) != PROTOCOL[k]}
    assert not changed, f"Protocol already registered and differs in {changed}."
    PROTOCOL["created_utc"] = old["created_utc"]
else:
    protocol_path.write_text(json.dumps(PROTOCOL, indent=2))
print("Protocol sha256:", hashlib.sha256(protocol_path.read_bytes()).hexdigest()[:16])

Protocol sha256: bbe8b3c073924910


## Data and windows (identical to the frozen pipeline)

In [3]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
P = PROTOCOL
t = np.load(ROOT / "artifacts/FD004_baseline_tensors.npz")
Xtr_raw, ytr, Xv_raw, yv = t["X_train"], t["y_train"].astype(np.float32), t["X_val"], t["y_val"].astype(np.float32)

def engine_ids(y):
    # Windows are stored engine by engine with RUL non-increasing inside an engine; an increase starts a new engine.
    ids = np.ones(len(y), int)
    ids[1:] = 1 + np.cumsum(np.diff(y) > 0)
    return ids

id_tr, id_v = engine_ids(ytr), engine_ids(yv)
assert id_tr.max() == 200 and id_v.max() == 49 and len(ytr) == 43696 and len(yv) == 10332

np.random.seed(42)
cal_engines = np.random.choice(np.unique(id_v), size=25, replace=False)
test_engines = np.setdiff1d(np.unique(id_v), cal_engines)
split_file = ROOT / "models/b6_variants/b6_cal_eval_split_ids.npz"
if split_file.exists():
    s = np.load(split_file)
    assert set(s["cal_engine_ids"]) == set(cal_engines) and set(s["eval_engine_ids"]) == set(test_engines)
    print("IID calibration/evaluation split matches the archived split file.")

def onehot(X):
    return np.concatenate([X[:, :, :14], np.eye(6, dtype=np.float32)[X[:, :, 14].astype(int)]], axis=-1).astype(np.float32)

train_engines = np.unique(id_tr)
fit_engines, ival_engines = train_test_split(train_engines, test_size=P["internal_validation"]["fraction_of_train_engines"],
                                             random_state=P["internal_validation"]["split_seed"])
mf, mi = np.isin(id_tr, fit_engines), np.isin(id_tr, ival_engines)
mc, mt = np.isin(id_v, cal_engines), np.isin(id_v, test_engines)
X_fit, y_fit = onehot(Xtr_raw[mf]), ytr[mf]
X_iv, y_iv = onehot(Xtr_raw[mi]), ytr[mi]
X_cal, y_cal, id_cal, reg_cal = onehot(Xv_raw[mc]), yv[mc], id_v[mc], Xv_raw[mc][:, -1, 14].astype(int)
X_test, y_test, id_test, reg_test = onehot(Xv_raw[mt]), yv[mt], id_v[mt], Xv_raw[mt][:, -1, 14].astype(int)
cyc_test = np.concatenate([np.arange((id_test == e).sum()) for e in np.unique(id_test)])  # window index within engine
assert len(y_cal) == 4909 and len(y_test) == 5423
print(f"Train engines {len(train_engines)} -> fit {len(fit_engines)} + internal validation {len(ival_engines)}")
print("fit", X_fit.shape, "| ival", X_iv.shape, "| cal", X_cal.shape, "| test", X_test.shape)
pd.concat([pd.DataFrame({"split": n, "engine_index": v}) for n, v in
           [("fit", fit_engines), ("internal_val", ival_engines), ("calibration", np.sort(cal_engines)), ("evaluation", test_engines)]]
          ).to_csv(OUT / "engine_split.csv", index=False)

IID calibration/evaluation split matches the archived split file.
Train engines 200 -> fit 160 + internal validation 40
fit (34983, 30, 20) | ival (8713, 30, 20) | cal (4909, 30, 20) | test (5423, 30, 20)


## Part B: train five seeds and check convergence (no evaluation data)

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Layer
from tensorflow.keras.models import Model
Q = P["model"]["quantiles"]

@tf.keras.utils.register_keras_serializable(package="fd004")
class FastTemporalAttention(Layer):
    def build(self, input_shape):
        d = input_shape[-1]
        self.W = self.add_weight(name="W", shape=(d, d), initializer="glorot_uniform")
        self.b = self.add_weight(name="b", shape=(d,), initializer="zeros")
        self.v = self.add_weight(name="v", shape=(d, 1), initializer="glorot_uniform")
        super().build(input_shape)
    def call(self, x):
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        return tf.reduce_sum(x * tf.nn.softmax(tf.tensordot(e, self.v, axes=1), axis=1), axis=1)

@tf.keras.utils.register_keras_serializable(package="fd004")
def multi_quantile_pinball_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
    return tf.reduce_mean(tf.add_n([tf.maximum(q * (y_true - y_pred[:, i:i+1]), (q - 1.0) * (y_true - y_pred[:, i:i+1]))
                                    for i, q in enumerate(Q)]))

def build_model():
    inp = Input(shape=(30, 20), name="input_window")
    h = Dense(18, use_bias=False, name="projection")(inp)
    h = Bidirectional(LSTM(64, return_sequences=True), name="bilstm")(h)
    h = FastTemporalAttention(name="temporal_attention")(h)
    m = Model(inp, Dense(3, name="quantiles")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=multi_quantile_pinball_loss)
    return m

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s); os.environ["PYTHONHASHSEED"] = str(s)

const_iv = float(sum(np.mean(np.maximum(q * (y_iv - np.quantile(y_fit, q)), (q - 1) * (y_iv - np.quantile(y_fit, q)))) for q in Q))
print(f"Constant-quantile loss on internal validation: {const_iv:.3f}")
(OUT / "environment.json").write_text(json.dumps({"python": platform.python_version(), "platform": platform.platform(),
    "tensorflow": tf.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}, indent=2))

tr = P["training_rule"]; train_log = []
for seed in P["model_seeds"]:
    tf.keras.backend.clear_session(); set_seed(seed); model = build_model()
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=tr["patience"], start_from_epoch=tr["start_from_epoch"],
                                            restore_best_weights=True)
    t0 = time.time()
    hist = model.fit(X_fit, y_fit, validation_data=(X_iv, y_iv), epochs=tr["max_epochs"], batch_size=256, shuffle=True,
                     callbacks=[stop], verbose=2)
    h = pd.DataFrame(hist.history); h.insert(0, "epoch", np.arange(1, len(h) + 1))
    h.to_csv(OUT / "histories" / f"seed{seed}_history.csv", index=False)
    model.save(OUT / "models" / f"{SLUG}_rerun_seed{seed}.keras")
    best = float(h.val_loss.min())
    train_log.append({"seed": seed, "epochs_run": len(h), "best_epoch": int(h.loc[h.val_loss.idxmin(), "epoch"]),
                      "best_val_loss": best, "constant_val_loss": const_iv, "ratio": best / const_iv,
                      "converged": best <= P["convergence_check"]["ratio"] * const_iv, "minutes": (time.time() - t0) / 60})
    print(train_log[-1])
train_log = pd.DataFrame(train_log); train_log.to_csv(OUT / "convergence_check.csv", index=False)
print(train_log.round(3).to_string(index=False))

Constant-quantile loss on internal validation: 28.908


Epoch 1/150


137/137 - 10s - 71ms/step - loss: 111.9628 - val_loss: 95.6733


Epoch 2/150


137/137 - 10s - 72ms/step - loss: 85.9970 - val_loss: 76.7721


...

Epoch 57/150


137/137 - 10s - 76ms/step - loss: 9.9807 - val_loss: 11.1550


Epoch 58/150


137/137 - 11s - 78ms/step - loss: 9.4984 - val_loss: 11.2540


{'seed': 1000, 'epochs_run': 58, 'best_epoch': 43, 'best_val_loss': 10.85485553741455, 'constant_val_loss': 28.908205032348633, 'ratio': 0.3754939307116383, 'converged': True, 'minutes': 10.237919346491497}
 seed  epochs_run  best_epoch  best_val_loss  constant_val_loss  ratio  converged  minutes
   42          46          31         11.529             28.908  0.399       True   13.514
  123          50          29         12.547             28.908  0.434       True   16.141
  456          47          32         15.235             28.908  0.527       True   14.585
  789         101          86         12.571             28.908  0.435       True   26.336
 1000          58          43         10.855             28.908  0.375       True   10.238


## Part C: evaluation (run only after Part B is saved)

In [5]:
assert (OUT / "convergence_check.csv").exists()
from scipy.stats import pearsonr, spearmanr
A = P["calibration"]["alpha"]

def cq(scores, a):
    n = len(scores)
    return float(np.quantile(scores, min(1.0, np.ceil((n + 1) * (1 - a)) / n), method="higher"))

def stage_of(r):
    return np.where(r > 80, "Early", np.where(r > 40, "Mid", "Late"))

rows, glob_rows, eng_rows, stage_rows, corr_rows, corrections = [], [], [], [], [], []
for seed in P["model_seeds"]:
    model = tf.keras.models.load_model(OUT / "models" / f"{SLUG}_rerun_seed{seed}.keras")
    pc = model.predict(X_cal, batch_size=512, verbose=0); pt = model.predict(X_test, batch_size=512, verbose=0)
    lc, uc, lt, ut = pc[:, 0], pc[:, 2], pt[:, 0], pt[:, 2]
    qw = cq(np.maximum(lc - y_cal, y_cal - uc), A)
    ql, qu = cq(lc - y_cal, A / 2), cq(y_cal - uc, A / 2)
    err = np.maximum(lc - y_cal, y_cal - uc)
    eq = np.array([np.quantile(err[id_cal == e], 0.80) for e in np.unique(id_cal)]); M = len(eq)
    qb = float(np.quantile(eq, min(1.0, np.ceil((M + 1) * (1 - A)) / M), method="higher"))
    rl, ru = np.zeros_like(lt), np.zeros_like(ut)
    for r in range(6):
        mcal, mt = reg_cal == r, reg_test == r
        a1, a2 = cq(lc[mcal] - y_cal[mcal], A / 2), cq(y_cal[mcal] - uc[mcal], A / 2)
        rl[mt], ru[mt] = lt[mt] - a1, ut[mt] + a2
        corrections.append({"seed": seed, "method": "Regime-conditional", "regime": r, "q_lower": a1, "q_upper": a2})
    corrections += [{"seed": seed, "method": "Window", "q": qw}, {"seed": seed, "method": "Asymmetric", "q_lower": ql, "q_upper": qu},
                    {"seed": seed, "method": "Balanced", "q": qb}]
    methods = {"Raw Quantiles": (lt, ut), "Window CQR": (lt - qw, ut + qw), "Asymmetric CQR": (lt - ql, ut + qu),
               "Regime-conditional Asym. CQR": (rl, ru), "Balanced Engine CQR": (lt - qb, ut + qb)}
    df = pd.DataFrame({"seed": seed, "engine": id_test, "cycle": cyc_test, "endpoint_regime": reg_test, "true_RUL": y_test,
                       "q10": lt, "q50": pt[:, 1], "q90": ut, "stage": stage_of(y_test)})
    df["error_q50_minus_true"] = df.q50 - df.true_RUL
    for name, (lo, hi) in methods.items():
        lo, hi = np.maximum(0, lo), np.maximum(0, hi)
        k = name.split()[0].lower().replace("-", "_")
        df[f"{k}_lower"], df[f"{k}_upper"] = lo, hi
        cov = (y_test >= lo) & (y_test <= hi)
        wink = (hi - lo) + (2 / A) * (lo - y_test) * (y_test < lo) + (2 / A) * (y_test - hi) * (y_test > hi)
        ec = pd.Series(cov).groupby(id_test).mean() * 100
        glob_rows.append({"seed": seed, "method": name, "RMSE": float(np.sqrt(np.mean(df.error_q50_minus_true ** 2))),
                          "MAE": float(df.error_q50_minus_true.abs().mean()), "PICP": 100 * cov.mean(), "MPIW": float((hi - lo).mean()),
                          "Winkler": float(wink.mean()), "lower_violation": 100 * (y_test < lo).mean(), "upper_violation": 100 * (y_test > hi).mean(),
                          "eng_mean": ec.mean(), "eng_median": ec.median(), "eng_p10": np.percentile(ec, 10), "eng_p25": np.percentile(ec, 25),
                          "eng_worst": ec.min(), "below80": int((ec < 80).sum()), "below70": int((ec < 70).sum())})
        df[f"{k}_covered"] = cov
    for e, g in df.groupby("engine"):
        eng_rows.append({"seed": seed, "engine": e, "n": len(g), "RMSE": float(np.sqrt(np.mean(g.error_q50_minus_true ** 2))),
                         "MAE": float(g.error_q50_minus_true.abs().mean()), "bias": float(g.error_q50_minus_true.mean()),
                         "asym_cov": 100 * g.asymmetric_covered.mean(), "bal_cov": 100 * g.balanced_covered.mean(),
                         "bal_below_lower": 100 * (g.true_RUL < g.balanced_lower).mean(), "bal_above_upper": 100 * (g.true_RUL > g.balanced_upper).mean(),
                         "mid_bal_cov": 100 * g.balanced_covered[g.stage == "Mid"].mean() if (g.stage == "Mid").any() else np.nan,
                         "dominant_regime": int(g.endpoint_regime.mode().iloc[0]), "dominant_share": float((g.endpoint_regime == g.endpoint_regime.mode().iloc[0]).mean())})
    for s_, g in df.groupby("stage"):
        stage_rows.append({"seed": seed, "stage": s_, "N": len(g), "RMSE": float(np.sqrt(np.mean(g.error_q50_minus_true ** 2))),
                           "signed_bias": float(g.error_q50_minus_true.mean()), "asym_cov": 100 * g.asymmetric_covered.mean(),
                           "bal_cov": 100 * g.balanced_covered.mean()})
    pe = pd.DataFrame([r for r in eng_rows if r["seed"] == seed])
    corr_rows.append({"seed": seed, "pearson_r": pearsonr(pe.RMSE, pe.bal_cov)[0], "pearson_p": pearsonr(pe.RMSE, pe.bal_cov)[1],
                      "spearman_rho": spearmanr(pe.RMSE, pe.bal_cov)[0],
                      "problem_engines": " ".join(map(str, sorted(pe.engine[pe.bal_cov < 80])))})
    rows.append(df)

pred = pd.concat(rows, ignore_index=True); pred.to_csv(OUT / "test_predictions_rowlevel.csv", index=False)
glob = pd.DataFrame(glob_rows).merge(train_log[["seed", "converged"]], on="seed"); glob.to_csv(OUT / "global_by_seed.csv", index=False)
per_engine = pd.DataFrame(eng_rows); per_engine.to_csv(OUT / "per_engine_by_seed.csv", index=False)
stages = pd.DataFrame(stage_rows); stages.to_csv(OUT / "life_stage_by_seed.csv", index=False)
pd.DataFrame(corr_rows).to_csv(OUT / "rmse_coverage_correlation_by_seed.csv", index=False)
pd.DataFrame(corrections).to_csv(OUT / "corrections_by_seed.csv", index=False)
print(glob[glob.seed == P["model_seeds"][0]].round(2).to_string(index=False))

 seed                       method  RMSE   MAE  PICP  MPIW  Winkler  lower_violation  upper_violation  eng_mean  eng_median  eng_p10  eng_p25  eng_worst  below80  below70  converged
   42                Raw Quantiles 22.02 13.81 72.49 37.58    67.91             8.92            18.59     71.73       71.66    49.31    56.57      38.67       14       11       True
   42                   Window CQR 22.02 13.81 82.76 41.47    67.89             6.77            10.47     81.72       80.21    63.62    69.86      51.93       12        6       True
   42               Asymmetric CQR 22.02 13.81 81.97 41.85    68.50             5.96            12.08     81.37       82.27    60.44    68.35      45.30       10        7       True
   42 Regime-conditional Asym. CQR 22.02 13.81 82.06 42.34    68.65             6.05            11.89     81.42       82.40    60.88    70.39      45.30       10        6       True
   42          Balanced Engine CQR 22.02 13.81 93.05 70.53    79.99             2.38      

In [6]:
# Engine-cluster bootstrap and five-seed summary
boot = []
for seed in P["model_seeds"]:
    g = pred[pred.seed == seed]
    agg = g.groupby("engine").agg(n=("true_RUL", "size"), ca=("asymmetric_covered", "sum"), cb=("balanced_covered", "sum"))
    agg["wa"] = (g.asymmetric_upper - g.asymmetric_lower).groupby(g.engine).sum()
    agg["wb"] = (g.balanced_upper - g.balanced_lower).groupby(g.engine).sum()
    n = agg.n.to_numpy(); rng = np.random.default_rng(42); idx = rng.integers(0, len(n), size=(10000, len(n))); N = n[idx].sum(1)
    covA, covB = 100 * agg.ca.to_numpy() / n, 100 * agg.cb.to_numpy() / n
    S = {"asym_picp": (100 * agg.ca.sum() / n.sum(), 100 * agg.ca.to_numpy()[idx].sum(1) / N),
         "bal_picp": (100 * agg.cb.sum() / n.sum(), 100 * agg.cb.to_numpy()[idx].sum(1) / N),
         "asym_mpiw": (agg.wa.sum() / n.sum(), agg.wa.to_numpy()[idx].sum(1) / N),
         "bal_mpiw": (agg.wb.sum() / n.sum(), agg.wb.to_numpy()[idx].sum(1) / N),
         "asym_eng_mean": (covA.mean(), covA[idx].mean(1)), "bal_eng_mean": (covB.mean(), covB[idx].mean(1)),
         "asym_eng_p10": (np.percentile(covA, 10), np.percentile(covA[idx], 10, axis=1)),
         "bal_eng_p10": (np.percentile(covB, 10), np.percentile(covB[idx], 10, axis=1)),
         "asym_eng_worst": (covA.min(), covA[idx].min(1)), "bal_eng_worst": (covB.min(), covB[idx].min(1))}
    S["bal_minus_asym_picp"] = (S["bal_picp"][0] - S["asym_picp"][0], S["bal_picp"][1] - S["asym_picp"][1])
    for k, (pt_, v) in S.items():
        boot.append({"seed": seed, "stat": k, "point": pt_, "ci_low": np.percentile(v, 2.5), "ci_high": np.percentile(v, 97.5)})
pd.DataFrame(boot).to_csv(OUT / "bootstrap_by_seed.csv", index=False)

five = glob.groupby("method")[["PICP", "MPIW", "Winkler", "RMSE", "MAE", "below80", "eng_worst"]].agg(["mean", "std"])
five.columns = [f"{a}_{b}" for a, b in five.columns]; five.to_csv(OUT / "five_seed_summary.csv")
stages.groupby("stage")[["RMSE", "signed_bias", "bal_cov", "asym_cov"]].agg(["mean", "std"]).to_csv(OUT / "life_stage_five_seed_summary.csv")
print(five.round(2).to_string())
manifest = {str(p.relative_to(OUT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.rglob("*")) if p.is_file()}
(OUT / "manifest_sha256.json").write_text(json.dumps(manifest, indent=2)); print("Done:", len(manifest), "files")

                              PICP_mean  PICP_std  MPIW_mean  MPIW_std  Winkler_mean  Winkler_std  RMSE_mean  RMSE_std  MAE_mean  MAE_std  below80_mean  below80_std  eng_worst_mean  eng_worst_std
method                                                                                                                                                                                             
Asymmetric CQR                    82.65      1.50      48.99      7.60         75.82         6.81      22.74      1.11     14.22     0.62          11.0         1.22           50.17           9.13
Balanced Engine CQR               92.80      0.84      78.85      7.90         88.15         6.99      22.74      1.11     14.22     0.62           3.2         1.10           72.63           1.27
Raw Quantiles                     69.67      4.10      35.68      2.50         78.71         9.88      22.74      1.11     14.22     0.62          16.2         3.03           33.71           6.80
Regime-conditional A